In [ ]:
!pip install mne tensorflow scikit-learn numpy pandas matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import numpy as np
import pandas as pd
import mne
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras import layers, models, constraints

DATA_DIR = "/content/drive/MyDrive/EEG"

FS = 160
EPOCH_START = 0.0
EPOCH_END = 4.0

IMAGERY_RUNS = {
    4: {"T1": "left_fist", "T2": "right_fist"},
    8: {"T1": "left_fist", "T2": "right_fist"},
    12: {"T1": "left_fist", "T2": "right_fist"},

    6: {"T1": "both_fists", "T2": "both_feet"},
    10: {"T1": "both_fists", "T2": "both_feet"},
    14: {"T1": "both_fists", "T2": "both_feet"},
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_subject_epochs(subject_folder):
    X = []
    y = []

    subject_path = os.path.join(DATA_DIR, subject_folder)

    for run_num, label_map in IMAGERY_RUNS.items():
        file_name = f"{subject_folder}R{run_num:02d}.edf"
        file_path = os.path.join(subject_path, file_name)

        if not os.path.exists(file_path):
            print(f"Missing file: {file_path}")
            continue

        raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)

        raw.pick("eeg")

        # Keep same filter as your old CNN for fair comparison
        raw.filter(1.0, 40.0, verbose=False)

        events, event_id = mne.events_from_annotations(raw, verbose=False)

        usable_event_id = {
            code: event_id[code]
            for code in ["T1", "T2"]
            if code in event_id
        }

        if len(usable_event_id) == 0:
            del raw
            gc.collect()
            continue

        epochs = mne.Epochs(
            raw,
            events,
            event_id=usable_event_id,
            tmin=EPOCH_START,
            tmax=EPOCH_END,
            baseline=None,
            preload=True,
            reject_by_annotation=True,
            verbose=False
        )

        data = epochs.get_data().astype(np.float32)

        reverse_event_id = {v: k for k, v in usable_event_id.items()}

        for i, event_code in enumerate(epochs.events[:, -1]):
            event_name = reverse_event_id[event_code]

            if event_name not in label_map:
                continue

            X.append(data[i])
            y.append(label_map[event_name])

        del raw
        del epochs
        del data
        gc.collect()

    return X, y

In [ ]:
all_X = []
all_y = []
all_subject_ids = []

subjects = sorted([
    folder for folder in os.listdir(DATA_DIR)
    if folder.startswith("S") and os.path.isdir(os.path.join(DATA_DIR, folder))
])

print("Found subjects:", subjects[:10], "...")

for subject in subjects:
    print("Loading", subject)
    X_subject, y_subject = load_subject_epochs(subject)

    all_X.extend(X_subject)
    all_y.extend(y_subject)
    all_subject_ids.extend([subject] * len(y_subject))

min_time_len = min(epoch.shape[1] for epoch in all_X)
all_X_fixed = [epoch[:, :min_time_len] for epoch in all_X]

X = np.stack(all_X_fixed).astype(np.float32)
y = np.array(all_y)
subject_ids = np.array(all_subject_ids)

del all_X
del all_X_fixed
gc.collect()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Subject IDs shape:", subject_ids.shape)
print("Labels:", sorted(set(y)))
print("X dtype:", X.dtype)

Found subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010'] ...
Loading S001
Loading S002
Loading S003
Loading S004
Loading S005
Loading S006
Loading S007
Loading S008
Loading S009
Loading S010
Loading S011
Loading S012
Loading S013
Loading S014
Loading S015
Loading S016
Loading S017
Loading S018
Loading S019
Loading S020
Loading S021
Loading S022
Loading S023
Loading S024
Loading S025
Loading S026
Loading S027
Loading S028
Loading S029
Loading S030
Loading S031
Loading S032
Loading S033
Loading S034
Loading S035
Loading S036
Loading S037
Loading S038
Loading S039
Loading S040
Loading S041
Loading S042
Loading S043
Loading S044
Loading S045
Loading S046
Loading S047
Loading S048
Loading S049
Loading S050
Loading S051
Loading S052
Loading S053
Loading S054
Loading S055
Loading S056
Loading S057
Loading S058
Loading S059
Loading S060
Loading S061
Loading S062
Loading S063
Loading S064
Loading S065
Loading S066
Loading S067
Loading S068
Loading S069
Lo

/tmp/ipykernel_13088/1863504658.py:15: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
/tmp/ipykernel_13088/1863504658.py:15: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
/tmp/ipykernel_13088/1863504658.py:15: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
/tmp/ipykernel_13088/1863504658.py:15: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
/tmp/ipykernel_13088/1863504658.py:15: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
/tmp/ipykernel_13088/1863504658.py:15: Runtim

Loading S101
Loading S102
Loading S103
Loading S104
Loading S105
Loading S106
Loading S107
Loading S108
Loading S109
X shape: (9795, 64, 513)
y shape: (9795,)
Subject IDs shape: (9795,)
Labels: [np.str_('both_feet'), np.str_('both_fists'), np.str_('left_fist'), np.str_('right_fist')]
X dtype: float32


In [ ]:
X = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-8)
X = X.astype(np.float32)

X = X[..., np.newaxis]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Encoded labels:")
for i, label in enumerate(label_encoder.classes_):
    print(i, label)

print("X shape after adding channel dimension:", X.shape)
print("X dtype:", X.dtype)

Encoded labels:
0 both_feet
1 both_fists
2 left_fist
3 right_fist
X shape after adding channel dimension: (9795, 64, 513, 1)
X dtype: float32


In [ ]:
gss_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

trainval_idx, test_idx = next(
    gss_test.split(
        X,
        y_encoded,
        groups=subject_ids
    )
)

X_trainval = X[trainval_idx]
y_trainval = y_encoded[trainval_idx]
groups_trainval = subject_ids[trainval_idx]

X_test = X[test_idx]
y_test = y_encoded[test_idx]
groups_test = subject_ids[test_idx]

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=123
)

train_idx, val_idx = next(
    gss_val.split(
        X_trainval,
        y_trainval,
        groups=groups_trainval
    )
)

X_train = X_trainval[train_idx]
y_train = y_trainval[train_idx]
groups_train = groups_trainval[train_idx]

X_val = X_trainval[val_idx]
y_val = y_trainval[val_idx]
groups_val = groups_trainval[val_idx]

print("\n===== DATA SHAPES =====")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\n===== SUBJECT COUNTS =====")
print("Train subjects:", len(set(groups_train)))
print("Validation subjects:", len(set(groups_val)))
print("Test subjects:", len(set(groups_test)))

train_val_overlap = set(groups_train).intersection(set(groups_val))
train_test_overlap = set(groups_train).intersection(set(groups_test))
val_test_overlap = set(groups_val).intersection(set(groups_test))

print("\n===== SUBJECT OVERLAP CHECK =====")
print("Train/Val overlap:", train_val_overlap)
print("Train/Test overlap:", train_test_overlap)
print("Val/Test overlap:", val_test_overlap)

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("\nNo subject leakage detected.")


===== DATA SHAPES =====
Train: (6198, 64, 513, 1)
Validation: (1628, 64, 513, 1)
Test: (1969, 64, 513, 1)

===== SUBJECT COUNTS =====
Train subjects: 69
Validation subjects: 18
Test subjects: 22

===== SUBJECT OVERLAP CHECK =====
Train/Val overlap: set()
Train/Test overlap: set()
Val/Test overlap: set()

No subject leakage detected.


In [ ]:
tf.keras.backend.clear_session()
gc.collect()

num_classes = len(label_encoder.classes_)

def build_eegnet(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    # Temporal filtering
    x = layers.Conv2D(
        filters=16,
        kernel_size=(1, 64),
        padding="same",
        use_bias=False
    )(inputs)

    x = layers.BatchNormalization()(x)

    # Spatial filtering across all EEG channels
    x = layers.DepthwiseConv2D(
        kernel_size=(input_shape[0], 1),
        depth_multiplier=2,
        use_bias=False,
        depthwise_constraint=constraints.max_norm(1.0)
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)

    x = layers.AveragePooling2D(
        pool_size=(1, 4)
    )(x)

    x = layers.Dropout(0.25)(x)

    # Separable convolution for compact EEG feature extraction
    x = layers.SeparableConv2D(
        filters=32,
        kernel_size=(1, 16),
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)

    x = layers.AveragePooling2D(
        pool_size=(1, 8)
    )(x)

    x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        kernel_constraint=constraints.max_norm(0.5)
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs
    )

    return model


model = build_eegnet(
    input_shape=X_train.shape[1:],
    num_classes=num_classes
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 513, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 513, 16)    │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 513, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 1, 513, 32)     │         2,048 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 513, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1, 513, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 1, 128, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 128, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 1, 128, 32)     │         1,536 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 128, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1, 128, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 1, 16, 32)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 16, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,980 (27.27 KB)

 Trainable params: 6,820 (26.64 KB)

 Non-trainable params: 160 (640.00 B)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    shuffle=True
)

Epoch 1/80
194/194 ━━━━━━━━━━━━━━━━━━━━ 350s 2s/step - accuracy: 0.4590 - loss: 1.2000 - val_accuracy: 0.5006 - val_loss: 1.2208 - learning_rate: 0.0010
Epoch 2/80
194/194 ━━━━━━━━━━━━━━━━━━━━ 347s 2s/step - accuracy: 0.5486 - loss: 1.0613 - val_accuracy: 0.5215 - val_loss: 1.1615 - learning_rate: 0.0010
Epoch 3/80
194/194 ━━━━━━━━━━━━━━━━━━━━ 380s 2s/step - accuracy: 0.5705 - loss: 1.0278 - val_accuracy: 0.5442 - val_loss: 1.1418 - learning_rate: 0.0010
Epoch 4/80
194/194 ━━━━━━━━━━━━━━━━━━━━ 349s 2s/step - accuracy: 0.5866 - loss: 0.9996 - val_accuracy: 0.5510 - val_loss: 1.1231 - learning_rate: 0.0010
Epoch 5/80
194/194 ━━━━━━━━━━━━━━━━━━━━ 339s 2s/step - accuracy: 0.6012 - loss: 0.9789 - val_accuracy: 0.5584 - val_loss: 1.1086 - learning_rate: 0.0010
Epoch 6/80
194/194 ━━━━━━━━━━━━━━━━━━━━ 387s 2s/step - accuracy: 0.6070 - loss: 0.9768 - val_accuracy: 0.5596 - val_loss: 1.1065 - learning_rate: 0.0010
Epoch 7/80
194/194 ━━━━━━━━━━━━━━━━━━━━ 333s 2s/step - accuracy: 0.6194 - loss: 0.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test accuracy:", test_acc)
print("Test loss:", test_loss)

62/62 ━━━━━━━━━━━━━━━━━━━━ 17s 269ms/step - accuracy: 0.5998 - loss: 0.9752
Test accuracy: 0.5997968316078186
Test loss: 0.975242555141449


In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in label_encoder.classes_],
    columns=[f"pred_{label}" for label in label_encoder.classes_]
)

print(cm_df)

62/62 ━━━━━━━━━━━━━━━━━━━━ 17s 269ms/step
              precision    recall  f1-score   support

   both_feet       0.53      0.60      0.56       494
  both_fists       0.54      0.60      0.57       490
   left_fist       0.71      0.60      0.65       497
  right_fist       0.65      0.60      0.63       488

    accuracy                           0.60      1969
   macro avg       0.61      0.60      0.60      1969
weighted avg       0.61      0.60      0.60      1969

                 pred_both_feet  pred_both_fists  pred_left_fist  \
true_both_feet              297               77              55   
true_both_fists             105              292              34   
true_left_fist               80               87             298   
true_right_fist              77               83              34   

                 pred_right_fist  
true_both_feet                65  
true_both_fists               59  
true_left_fist                32  
true_right_fist              294  


In [ ]:
model.save("/content/drive/MyDrive/eeg_motor_imagery_eegnet_stronger.keras")

np.save("/content/drive/MyDrive/eeg_label_classes.npy", label_encoder.classes_)